In [17]:
import os
import pprint
## Get the conll file from https://github.com/UniversalDependencies/UD_English-PUD/blob/master/en_pud-ud-test.conllu

os.listdir('assets')
cfile = open('assets/en_pud-ud-test.conllu')
crows = cfile.readlines()
# cfile.read()
# print(cfile.read())

In [44]:
import re
for i,r in enumerate(crows):
    if  re.search(r'^[0-9]', r):
        crow = r.split('\t')
        if crow[1] == 'that':
            if crows[i+1].split('\t')[3] == "PRON": 
                print(f"{crows[i-2]} , {crows[i-1]} , {crows[i]}, {crows[i+1]} {crows[i+2]} ,")
# crows[:10]    

3	study	study	NOUN	NN	Number=Sing	4	nsubj	4:nsubj	_
 , 4	estimates	estimate	VERB	VBZ	Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin	0	root	0:root	_
 , 5	that	that	SCONJ	IN	_	8	mark	8:mark	_
, 6	it	it	PRON	PRP	Case=Nom|Gender=Neut|Number=Sing|Person=3|PronType=Prs	8	expl	8:expl	_
 7	would	would	AUX	MD	VerbForm=Fin	8	aux	8:aux	_
 ,
11	he	he	PRON	PRP	Case=Nom|Gender=Masc|Number=Sing|Person=3|PronType=Prs	12	nsubj	12:nsubj	_
 , 12	felt	feel	VERB	VBD	Mood=Ind|Tense=Past|VerbForm=Fin	4	acl:relcl	4:acl:relcl	_
 , 13	that	that	SCONJ	IN	_	16	mark	16:mark	_
, 14	they	they	PRON	PRP	Case=Nom|Number=Plur|Person=3|PronType=Prs	16	nsubj	16:nsubj	_
 15	should	should	AUX	MD	VerbForm=Fin	16	aux	16:aux	_
 ,
2	's	be	AUX	VBZ	Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin	3	cop	3:cop	_
 , 3	fantastic	fantastic	ADJ	JJ	Degree=Pos	0	root	0:root	_
 , 4	that	that	SCONJ	IN	_	6	mark	6:mark	_
, 5	they	they	PRON	PRP	Case=Nom|Number=Plur|Person=3|PronType=Prs	6	nsubj	6:nsubj	_
 6	got	get	VERB	VBD	Mood=I

In [ ]:
from stanza.utils.conll import CoNLL
import tarfile
import os
import argparse
args = argparse.ArgumentParser()
args.add_argument('--input', type=str)
args.add_argument('--output', type=str)
args = args.parse_args()
def get_file_list(path):
    """
    Get a list of files in the given directory.
    """
    file_list = []
    for root, dirs, files in os.walk(path):
        for file in files:
            if file.endswith('.conll'):
                file_list.append(os.path.join(root, file))
    return file_list

# Extract the tar file
files = get_file_list(args.input)
with open(args.output, 'w', encoding='utf-8') as outfile:
    outfile.write(f'FILEPATH\tTARGET\tNPI\tNPI_POS\tDEPREL\tMAIN\tSUB\tSENTENCE\n')
    for f in files:
        doc = CoNLL.conll2doc(f)
        out = []
        for sent in doc.sentences:
            for i, word in enumerate(sent.words):
                #Check if the word is a subordinating conjunction and not a case marker
                if word.text.lower() in ["bis", "bevor", 'ehe'] and word.deprel != "case" and word.pos != "NOUN":
                    subsent = sent.words[i:]
                    for subi,subword in enumerate(subsent):
                        try:
                            #check if nicht and 'word' depends from the same head and if the next word is not 'mehr'
                            if subword.text == "nicht" and subword.head == word.head and subsent[subi+1].text != "mehr": 
                                # we store the sub sentence 
                                subsent = " ".join(w.text for w in subsent)
                                # we store the main sentence
                                main_sent = ' '.join(w.text for w in sent.words[:i])
                                print(f'In file: {f}\nMAIN:__{main_sent}\n___SUB: {subsent}')
                                # write target, deprel, main and sub to the output file
                                outfile.write(f'{f}\t{word.text}\t{subword.text}\t{subword.pos}\t{word.deprel}\t{main_sent}\t{subsent}\t{sent.text}\n')
                            elif subword.text in ["kein", 'keine', 'keiner', 'keinem', 'keinen', 'keines']:
                                # we store the sub sentence 
                                subsent = " ".join(w.text for w in subsent)
                                # we store the main sentence
                                main_sent = ' '.join(w.text for w in sent.words[:i])
                                print(f'In file: {f}\n{subword.text}__MAIN:__{main_sent}\n___SUB: {subsent}')
                                # write target, deprel, main and sub to the output file
                                outfile.write(f'{f}\t{word.text}\t{subword.text}\t{subword.pos}\t{word.deprel}\t{main_sent}\t{subsent}\t{sent.text}\n')

                        except AttributeError:
                            print(f, sent.text)

                        


In [49]:
import gensim.downloader as api
from gensim.models import Word2Vec

# 1. Download and load the text8 dataset
# This returns an iterable: you don't load the whole 100MB into RAM at once
text8 = api.load("text8")

# 2. Initialize and train the model
# vector_size: dimensionality of the word vectors
# window: maximum distance between current and predicted word
# min_count: ignore words with total frequency lower than this
# workers: number of CPU threads to use
model = Word2Vec(sentences=text8, vector_size=100, window=5, min_count=5, workers=4)

# result = model.wv.most_similar("university")
# print("Words most similar to 'university':")
# for word, score in result:
#     print(f"{word}: {score:.4f}")

# 4. Save the model for later use


In [58]:
model.save("text8_word2vec.model")

In [52]:
len([word for text in text8 for word in text])

17005207

In [55]:
# "DOCTOR" - "MAN" + "WOMAN"

In [57]:
model.wv.most_similar(positive=["man", "black"], negative=["white"])

[('creature', 0.6035770773887634),
 ('woman', 0.595797061920166),
 ('evil', 0.5630866289138794),
 ('galactus', 0.550292432308197),
 ('hero', 0.5408021211624146),
 ('monster', 0.5397578477859497),
 ('vicious', 0.531773030757904),
 ('girl', 0.5308148264884949),
 ('immortal', 0.5237588286399841),
 ('demon', 0.5203119516372681)]

In [ ]:
import os
import datetime
import requests
import pandas as pd
import optuna
import logging
import mlflow
import argparse
from gensim.models import Word2Vec, FastText
from scipy.stats import spearmanr
from optuna.integration.mlflow import MLflowCallback

# Optional: Disable gensim info logging to keep output clean during optimization
logging.getLogger("gensim").setLevel(logging.ERROR)

def parse_args():
    parser = argparse.ArgumentParser(description="Train Word2Vec or FastText with Optuna optimization.")
    parser.add_argument(
        "--model", 
        type=str, 
        choices=["word2vec", "fasttext"],
        default="word2vec",
        help="Type of embedding model to train."
    )
    parser.add_argument(
        "--input", 
        type=str, 
        default='/data/users/luducceschi_eurac/kst/final_corpus_for_w2v_training/DE_corpus_2026-01-30_10-32-31_lemma_nopunct_ner_nonumber_nostopwords_sample.txt',
        help="Path to the input corpus text file."
    )
    parser.add_argument(
        "--db_dir", 
        type=str, 
        default='/data/users/luducceschi_eurac/kst/training',
        help="Directory to save Optuna and MLflow databases."
    )
    parser.add_argument(
        "--model_dir", 
        type=str, 
        default='/data/users/luducceschi_eurac/kst/models',
        help="Directory to save the final trained models."
    )
    parser.add_argument(
        "--tag", 
        type=str, 
        default=None,
        help="Unique tag for this run (e.g. timestamp or JobID). If None, a timestamp is used."
    )
    parser.add_argument(
        "--no_weight", 
        action="store_true",
        help="Disable coverage weighting (use raw correlation as optimization objective)."
    )
    parser.add_argument(
        "--trials", 
        type=int, 
        default=50,
        help="Number of optimization trials."
    )

    return parser.parse_args()

ARGS = parse_args()

class MySentences(object):
    def __init__(self, filename):
        self.filename = filename

    def __iter__(self):
        for line in open(self.filename, encoding="utf-8"):
            yield line.strip().split()

sentencesDE = MySentences(ARGS.input) # a memory-friendly iterator


def load_men_de():
    """Downloads and parses the German MEN dataset."""
    url = 'https://raw.githubusercontent.com/cophi-wue/Word-Embeddings-in-the-Digital-Humanities/refs/heads/main/testsets/MEN_de/MEN_dataset_de_full.tsv'
    response = requests.get(url)
    # Skip header and split lines
    lines = [line.split('\t') for line in response.text.split('\n') if line.strip()][1:]
    
    word_pairs = []
    for line in lines:
        try:
            # Format: english_word1, english_word2, german_word1, german_word2, score, tag1, tag2
            g1, g2, score = line[2], line[3], float(line[4])
            word_pairs.append((g1, g2, score))
        except (IndexError, ValueError):
            continue
    return word_pairs
MEN_DATA = load_men_de()
print(MEN_DATA)
print(len(MEN_DATA))



def evaluate_model(model, men_data, apply_weight=True):
    """
    Calculates Spearman correlation between model and MEN scores.
    Optionally weights the score by coverage to reward models that know more words.
    """
    model_sims = []
    true_scores = []
    
    for w1, w2, score in men_data:
        if w1 in model.wv and w2 in model.wv:
            sim = model.wv.similarity(w1, w2)
            model_sims.append(sim)
            true_scores.append(score)
    
    n_found = len(model_sims)
    if n_found < 10: # Not enough overlapping words
        return 0.0, 0.0, 0.0 # score, correlation, coverage
    
    correlation, _ = spearmanr(model_sims, true_scores)
    coverage = n_found / len(men_data)
    
    if apply_weight:
        # Score = correlation * sqrt(coverage)
        score = correlation * (coverage ** 0.5)
    else:
        score = correlation
        
    return score, correlation, coverage

def objective(trial):
    # 1. Suggest Hyperparameters
    params = {
        "vector_size": trial.suggest_int("vector_size", 100, 300),
        "window": trial.suggest_int("window", 2, 12),
        "min_count": trial.suggest_int("min_count", 1, 10),
        "sg": trial.suggest_categorical("sg", [0, 1]), # 0=CBOW, 1=Skip-gram
        "alpha": trial.suggest_float("alpha", 0.01, 0.05),
        "epochs": trial.suggest_int("epochs", 5, 15),
        "sample": trial.suggest_float("sample", 1e-5, 1e-3, log=True),
        "hs": trial.suggest_categorical("hs", [0, 1]), # 1=Hierarchical Softmax, 0=Negative Sampling
        "workers": 4
    }
    
    # Negative sampling is only used if Hierarchical Softmax is 0
    if params["hs"] == 0:
        params["negative"] = trial.suggest_int("negative", 5, 20)
    else:
        params["negative"] = 0 # Explicitly set to 0 when hs is 1
    
    if ARGS.model == "fasttext":
        params["min_n"] = trial.suggest_int("min_n", 2, 4)
        params["max_n"] = trial.suggest_int("max_n", 5, 8)
    
    # 2. Train Model
    if ARGS.model == "word2vec":
        model = Word2Vec(sentencesDE, **params)
    else:
        model = FastText(sentencesDE, **params)
    
    # 3. Evaluate
    apply_weight = not ARGS.no_weight
    score, correlation, coverage = evaluate_model(model, MEN_DATA, apply_weight=apply_weight)
    
    # 4. Log additional metrics to MLflow via Optuna user attributes
    trial.set_user_attr("raw_spearman", correlation)
    trial.set_user_attr("coverage", coverage)
    trial.set_user_attr("weighted", apply_weight)
    trial.set_user_attr("model_type", ARGS.model)
    
    return score


if __name__ == "__main__":
    # Ensure directories exist
    os.makedirs(ARGS.db_dir, exist_ok=True)
    os.makedirs(ARGS.model_dir, exist_ok=True)
    
    # Generate a unique tag (ensures timestamp is always included as requested)
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    if ARGS.tag:
        run_tag = f"{ARGS.tag}_{timestamp}"
    else:
        run_tag = timestamp
    
    # Optuna: Separate DB file for each run tag
    optuna_db_path = os.path.join(ARGS.db_dir, f"optuna_study_{run_tag}.db")
    optuna_storage = f"sqlite:///{optuna_db_path}"
    
    # Study name incorporates model selection and full tag (including timestamp)
    study_name = f"{ARGS.model}_optuna_{run_tag}"

    # MLflow: ONE shared DB file in the DB_DIR
    mlflow_db_path = os.path.join(ARGS.db_dir, "mlflow_master.db")
    mlflow_tracking_uri = f"sqlite:///{mlflow_db_path}"

    # Set MLflow tracking URI
    mlflow.set_tracking_uri(mlflow_tracking_uri)
    
    # Configure MLflow callback for Optuna
    mlflc = MLflowCallback(
        tracking_uri=mlflow_tracking_uri,
        metric_name="spearman_correlation",
    )

    print(f"--- Run Information ---")
    print(f"Model: {ARGS.model}")
    print(f"Tag: {run_tag}")
    print(f"Input Corpus: {ARGS.input}")
    print(f"Study Name: {study_name}")
    print(f"Optuna DB: {optuna_db_path}")
    print(f"MLflow Master DB: {mlflow_db_path}")
    print(f"Model Directory: {ARGS.model_dir}")
    print(f"Trials: {ARGS.trials}")
    print(f"-----------------------")

    study = optuna.create_study(
        study_name=study_name,
        storage=optuna_storage,
        direction="maximize",
        load_if_exists=True
    )

    # Run the optimization with MLflow callback
    study.optimize(objective, n_trials=ARGS.trials, callbacks=[mlflc])

    print("\nOptimization Finished!")
    print(f"Best Score: {study.best_value:.4f}")
    print("Best Parameters:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")

    # Retrain and save the best model
    print(f"\nRetraining the best {ARGS.model} model for {run_tag}...")
    with mlflow.start_run(run_name=f"best_model_{ARGS.model}_{run_tag}"):
        best_params = study.best_params.copy()
        best_params["workers"] = 4
        
        if ARGS.model == "word2vec":
            best_model = Word2Vec(sentencesDE, **best_params)
        else:
            best_model = FastText(sentencesDE, **best_params)
        
        # Save model in MODEL_DIR with tag and trial number
        model_filename = f"best_{ARGS.model}_{run_tag}_trial{study.best_trial.number}.model"
        model_path = os.path.join(ARGS.model_dir, model_filename)
        best_model.save(model_path)
        
        # Log params and model to MLflow
        mlflow.log_params(study.best_params)
        mlflow.log_artifact(model_path)
        
        print(f"Successfully saved best model to: {model_path}")
        print("Best model also logged as artifact in MLflow.")